In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드 (파일명: 스크리닝 최종.xlsx)
file_name = '스크리닝 최종.xlsx'

# Excel 파일의 모든 시트 이름을 가져옴
excel_file = pd.ExcelFile(file_name)
sheet_names = excel_file.sheet_names

# 만약 'Sheet1' 시트가 없으면 첫 번째 시트를 사용
if 'Sheet1' in sheet_names:
    df = pd.read_excel(file_name, sheet_name='Sheet1')
else:
    print(f"'Sheet1' 시트를 찾을 수 없습니다. 첫 번째 시트인 '{sheet_names[0]}'를 사용합니다.")
    df = pd.read_excel(file_name, sheet_name=sheet_names[0])

# 2. 데이터 전처리 및 컬럼 매핑
# Row 0: 종목명 / Row 3: 아이템명 / Row 5: 데이터 시작
stock_names = df.iloc[0, 1:].dropna().unique() # 종목명 리스트
date_col = df.columns[0] # 첫 번째 열(날짜)

# 스크리닝 결과를 담을 리스트
final_candidates = []

print(f"총 {len(stock_names)}개 종목 분석 중...\n")

for i, name in enumerate(stock_names):
    try:
        # 각 종목당 6개의 열을 순서대로 가져옴 (0~5번 인덱스)
        # 0: CFO / 1: 유동비율 / 2: 종가 / 3: 20일평균 / 4: 5일거래대금 / 5: PBR
        start_idx = 1 + (i * 6)
        cols = df.iloc[:, start_idx : start_idx + 6]

        # 날짜 열과 합치기
        temp_df = pd.concat([df[[date_col]], cols], axis=1)
        temp_df.columns = ['Date', 'CFO', 'Current_Ratio', 'Close', 'MA20', 'Volume', 'PBR']

        # Row 5부터 실제 숫자 데이터이므로 슬라이싱 후 수치형 변환
        temp_df = temp_df.iloc[5:].copy()
        for col in ['CFO', 'Current_Ratio', 'Close', 'MA20', 'Volume', 'PBR']:
            temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce')

        # 결측치 처리: 이전 자료로 채우기 (ffill), 하나도 없으면 드롭
        temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])

        if temp_df.empty:
            continue

        # --- [재무건전성 체크: 2024년 결산 기준] ---
        # 2024년 마지막 데이터 추출
        f_24 = temp_df[temp_df['Date'].astype(str).str.contains('2024')].iloc[-1]
        cfo_val = f_24['CFO']
        ratio_val = f_24['Current_Ratio']

        # --- [가격 및 수급 체크: 최신일자 기준] ---
        latest = temp_df.iloc[-1]
        curr_price = latest['Close']
        curr_ma20 = latest['MA20']
        curr_vol = latest['Volume']
        curr_pbr = latest['PBR']

        # 이격도 계산
        disparity = (curr_price / curr_ma20) * 100 if curr_ma20 > 0 else 0

        # --- [스크리닝 필터 적용] ---
        # 1. PBR: 5.0 ~ 15.0
        # 2. CFO: > 0 (24년 기준)
        # 3. 유동비율: > 100% (24년 기준)
        # 4. 이격도: 102% ~ 115%
        # 5. 5일 평균 거래대금: 50억 이상 (5,000,000,000원)

        if (5.0 <= curr_pbr <= 15.0) and \
           (cfo_val > 0) and \
           (ratio_val > 100) and \
           (102 <= disparity <= 115) and \
           (curr_vol >= 5000000000):

            final_candidates.append({
                '종목명': name,
                'PBR': round(curr_pbr, 2),
                'CFO(24)': f"{cfo_val:,.0f}",
                '유동비율(24)': f"{ratio_val:.2f}%",
                '이격도': f"{disparity:.2f}%",
                '거래대금(5일평균)': f"{curr_vol:,.0f}"
            })

    except Exception as e:
        continue

# 3. 최종 결과 출력 및 저장
result_df = pd.DataFrame(final_candidates)

if not result_df.empty:
    print("=== [KOSDAQ 150 근육형 대장주] 스크리닝 결과 ===")
    print(result_df)
    result_df.to_excel('스크리닝_최종_결과.xlsx', index=False)
    print("\n'스크리닝_최종_결과.xlsx' 파일로 저장 완료.")
else:
    print("조건을 모두 만족하는 종목이 없습니다. 필터값을 다시 점검해 보세요.")

총 150개 종목 분석 중...



/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_si

=== [KOSDAQ 150 근육형 대장주] 스크리닝 결과 ===
        종목명    PBR     CFO(24) 유동비율(24)      이격도       거래대금(5일평균)
0    리가켐바이오  12.55  78,476,381  469.58%  111.71%   95,378,784,360
1     원익IPS   6.85  78,280,504  281.76%  102.42%   51,277,716,840
2    이오테크닉스   8.70  55,361,111  926.38%  105.65%   37,860,464,150
3      HPSP  14.81  83,829,272  580.35%  114.48%  277,025,630,725
4      티씨케이   5.72  74,417,173  910.28%  110.78%   19,043,554,250
5  피에스케이홀딩스   5.57  66,625,216  466.08%  111.68%   23,416,137,150
6       네오셈   7.00  19,293,385  444.25%  106.03%   32,737,901,114
7      가온칩스  11.09  10,803,918  206.25%  103.17%    6,474,947,410

'스크리닝_최종_결과.xlsx' 파일로 저장 완료.


/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2851/1607218514.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_si

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드
file_name = '스크리닝 최종.xlsx'
df = pd.read_excel(file_name, sheet_name='Sheet1')

# 2. 기초 세팅
stock_names = df.iloc[0, 1:].dropna().unique()
date_col = df.columns[0]
final_candidates = []

print(f"재무 기준 완화 모드로 {len(stock_names)}개 종목 재분석 중...\n")

for i, name in enumerate(stock_names):
    try:
        # 각 종목당 6개 열 추출
        start_idx = 1 + (i * 6)
        cols = df.iloc[:, start_idx : start_idx + 6]

        temp_df = pd.concat([df[[date_col]], cols], axis=1)
        temp_df.columns = ['Date', 'CFO', 'Current_Ratio', 'Close', 'MA20', 'Volume', 'PBR']

        temp_df = temp_df.iloc[5:].copy()
        for col in ['CFO', 'Current_Ratio', 'Close', 'MA20', 'Volume', 'PBR']:
            temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce')

        # 결측치 처리 (이전 값 채우기)
        temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
        if temp_df.empty: continue

        # --- [재무 데이터: 24년 결산 기준] ---
        f_24 = temp_df[temp_df['Date'].astype(str).str.contains('2024')].iloc[-1]
        cfo_val = f_24['CFO']
        ratio_val = f_24['Current_Ratio']

        # --- [현재 데이터: 26년 3월 최신] ---
        latest = temp_df.iloc[-1]
        curr_price, curr_ma20, curr_vol, curr_pbr = latest['Close'], latest['MA20'], latest['Volume'], latest['PBR']
        disparity = (curr_price / curr_ma20) * 100 if curr_ma20 > 0 else 0

        # --- [수정된 스크리닝 필터 (완화 버전)] ---
        # 1. PBR: 5.0 ~ 15.0 (유지)
        # 2. CFO: > -20,000,000 (천원 단위 기준 약 -200억까지 허용하여 투자형 기업 포함)
        # 3. 유동비율: > 70% (기존 100%에서 70%로 완화)
        # 4. 이격도: 102% ~ 115% (유지)
        # 5. 거래대금: 50억 이상 (유지)

        cond_pbr = (5.0 <= curr_pbr <= 15.0)
        cond_cfo = (cfo_val > -20000000)      # 공격적 투자로 인한 마이너스 현금흐름 일부 허용
        cond_ratio = (ratio_val > 70)         # 단기 부채 부담이 조금 있더라도 포함
        cond_disparity = (102 <= disparity <= 115)
        cond_volume = (curr_vol >= 5000000000)

        if all([cond_pbr, cond_cfo, cond_ratio, cond_disparity, cond_volume]):
            final_candidates.append({
                '종목명': name,
                'PBR': round(curr_pbr, 2),
                'CFO(24)': f"{cfo_val:,.0f}",
                '유동비율(24)': f"{ratio_val:.2f}%",
                '이격도': f"{disparity:.2f}%",
                '거래대금(5일평균)': f"{curr_vol:,.0f}"
            })

    except Exception:
        continue

# 3. 결과 출력
result_df = pd.DataFrame(final_candidates)
if not result_df.empty:
    result_df = result_df.sort_values(by='이격도', ascending=False)
    print("=== [재무 완화] 스크리닝 결과 ===")
    print(result_df)
    result_df.to_excel('스크리닝_완화_결과.xlsx', index=False)
else:
    print("완화된 조건에서도 통과한 종목이 없습니다.")

재무 기준 완화 모드로 150개 종목 재분석 중...



/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_si

=== [재무 완화] 스크리닝 결과 ===
        종목명    PBR     CFO(24) 유동비율(24)      이격도       거래대금(5일평균)
3      HPSP  14.81  83,829,272  580.35%  114.48%  277,025,630,725
0    리가켐바이오  12.55  78,476,381  469.58%  111.71%   95,378,784,360
5  피에스케이홀딩스   5.57  66,625,216  466.08%  111.68%   23,416,137,150
4      티씨케이   5.72  74,417,173  910.28%  110.78%   19,043,554,250
8       네오셈   7.00  19,293,385  444.25%  106.03%   32,737,901,114
2    이오테크닉스   8.70  55,361,111  926.38%  105.65%   37,860,464,150
7   LS마린솔루션   5.50  -6,621,031  374.36%  103.95%   11,651,887,830
9      가온칩스  11.09  10,803,918  206.25%  103.17%    6,474,947,410
6    하나마이크론   6.20  40,971,030   99.42%  103.01%   58,210,164,505
1     원익IPS   6.85  78,280,504  281.76%  102.42%   51,277,716,840


/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df.ffill().dropna(subset=['Close', 'PBR'])
/tmp/ipykernel_2402/3926994400.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_si